# UAVDT BEV Project — Notebook 02
## Automatic BEV Homography Prototype

This notebook starts the BEV pipeline using the Drive outputs from Notebook 01.

Goal of this notebook:

1. Load a few UAVDT frames from Google Drive.
2. Automatically estimate an approximate image-to-BEV homography.
3. Warp the reference frame into a top-down-ish BEV view.
4. Save homography/debug outputs to Google Drive for later notebooks.

This is **not yet vehicle detection/tracking**. That will come later.

Default input sample folder:

```text
/Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/M1401_sample/images
```

Default output folder:

```text
/Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_02_bev_homography
```


In [ ]:
#@title 1. Set local project paths

from pathlib import Path
import json
import math
import shutil
import os
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
SAMPLE_IMAGES_DIR = PROJECT_DIR / 'work' / 'M1401_sample' / 'images'
OUTPUT_DIR = PROJECT_DIR / 'work' / 'notebook_02_bev_homography'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('SAMPLE_IMAGES_DIR:', SAMPLE_IMAGES_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Sample folder exists:', SAMPLE_IMAGES_DIR.exists())

if not SAMPLE_IMAGES_DIR.exists():
    raise RuntimeError('Sample images folder not found. Run Notebook 01 first, or update SAMPLE_IMAGES_DIR.')


In [ ]:
#@title 2. Imports and helper functions

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams['figure.figsize'] = (10, 6)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}


def list_images(folder: Path):
    return sorted([p for p in folder.iterdir() if p.suffix in IMAGE_EXTS])


def imread_rgb(path: Path):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise RuntimeError(f'Could not read image: {path}')
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def show_image(img_rgb, title=None, figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()


def save_rgb(path: Path, img_rgb):
    path.parent.mkdir(parents=True, exist_ok=True)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), img_bgr)


In [ ]:
#@title 3. Load and preview available sample frames

images = list_images(SAMPLE_IMAGES_DIR)
print('Number of sample images:', len(images))

if len(images) == 0:
    raise RuntimeError('No images found in SAMPLE_IMAGES_DIR.')

for i, p in enumerate(images[:10]):
    print(i, p.name)

# Pick a reference frame from the sample folder.
# If Notebook 01 created a small M1401 sample, index 0 is usually fine.
reference_image_index = 0 #@param {type:'integer'}
reference_image_index = max(0, min(reference_image_index, len(images) - 1))
ref_path = images[reference_image_index]
ref_rgb = imread_rgb(ref_path)

print('Reference image:', ref_path)
print('Reference image shape:', ref_rgb.shape)
show_image(ref_rgb, title=f'Reference frame: {ref_path.name}')


## Homography approach in this notebook

A true BEV homography cannot be determined from a single image without assumptions. This notebook uses a fast automatic approximation:

1. Detect edges.
2. Detect line segments with Hough transform.
3. Keep long non-horizontal line segments likely to be lane/road lines.
4. Estimate a road-direction vanishing point from line intersections.
5. Build a heuristic road quadrilateral around that vanishing point.
6. Map that quadrilateral to a rectangular BEV canvas.

This is intentionally lightweight. Later notebooks can replace the heuristic road quadrilateral with a segmentation model or map/lane prior.


In [ ]:
#@title 4. Automatic homography estimation functions


def line_intersection(l1, l2):
    x1, y1, x2, y2 = [float(v) for v in l1]
    x3, y3, x4, y4 = [float(v) for v in l2]

    den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(den) < 1e-6:
        return None

    px = ((x1*y2 - y1*x2) * (x3 - x4) - (x1 - x2) * (x3*y4 - y3*x4)) / den
    py = ((x1*y2 - y1*x2) * (y3 - y4) - (y1 - y2) * (x3*y4 - y3*x4)) / den
    return np.array([px, py], dtype=np.float32)


def filter_candidate_lines(lines, image_shape, min_length=60, angle_min_abs=25, angle_max_abs=155):
    h, w = image_shape[:2]
    candidates = []
    if lines is None:
        return candidates

    for l in lines[:, 0]:
        x1, y1, x2, y2 = [int(v) for v in l]
        dx = x2 - x1
        dy = y2 - y1
        length = float(np.hypot(dx, dy))
        if length < min_length:
            continue

        angle = float(np.degrees(np.arctan2(dy, dx)))
        if angle_min_abs < abs(angle) < angle_max_abs:
            candidates.append(np.array([x1, y1, x2, y2], dtype=np.float32))

    return candidates


def estimate_vanishing_point(candidate_lines, image_shape):
    h, w = image_shape[:2]
    intersections = []

    for i in range(len(candidate_lines)):
        for j in range(i + 1, len(candidate_lines)):
            p = line_intersection(candidate_lines[i], candidate_lines[j])
            if p is None:
                continue
            x, y = float(p[0]), float(p[1])
            # Broad bounds: allow vp above or near image, reject extreme outliers.
            if -1.0 * w < x < 2.0 * w and -1.0 * h < y < 1.2 * h:
                intersections.append(p)

    if len(intersections) == 0:
        return None, np.empty((0, 2), dtype=np.float32)

    pts = np.stack(intersections).astype(np.float32)

    # Robust-ish center: median is simple and fast.
    vp = np.median(pts, axis=0)
    return vp.astype(np.float32), pts


def build_road_quad_from_vp(vp, image_shape, top_y_ratio=0.20, bottom_y_ratio=0.98, bottom_left_ratio=0.20, bottom_right_ratio=0.88, top_width_ratio=0.18):
    h, w = image_shape[:2]

    if vp is None:
        # Fallback to image-center prior.
        vp = np.array([0.5 * w, 0.25 * h], dtype=np.float32)

    vp_x = float(np.clip(vp[0], 0.15 * w, 0.85 * w))
    vp_y = float(vp[1])

    # Top of visible road is usually below the vanishing point but not too low.
    top_y_default = top_y_ratio * h
    top_y = int(np.clip(max(top_y_default, vp_y + 0.06 * h), 0.12 * h, 0.55 * h))
    bottom_y = int(bottom_y_ratio * h)

    top_half_width = top_width_ratio * w

    src = np.float32([
        [vp_x - top_half_width, top_y],
        [vp_x + top_half_width, top_y],
        [bottom_right_ratio * w, bottom_y],
        [bottom_left_ratio * w, bottom_y],
    ])

    # Clip to image bounds.
    src[:, 0] = np.clip(src[:, 0], 0, w - 1)
    src[:, 1] = np.clip(src[:, 1], 0, h - 1)
    return src


def estimate_auto_bev_homography(img_rgb, bev_width=600, bev_height=1000, debug=True):
    h, w = img_rgb.shape[:2]

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 70, 170)

    # Region of interest: ignore the very top and far sides.
    roi = np.zeros_like(edges)
    roi[int(0.10 * h):h, int(0.10 * w):int(0.95 * w)] = 255
    edges_roi = cv2.bitwise_and(edges, roi)

    lines = cv2.HoughLinesP(
        edges_roi,
        rho=1,
        theta=np.pi / 180,
        threshold=60,
        minLineLength=50,
        maxLineGap=25,
    )

    candidates = filter_candidate_lines(lines, img_rgb.shape, min_length=60)
    vp, intersections = estimate_vanishing_point(candidates, img_rgb.shape)
    src_quad = build_road_quad_from_vp(vp, img_rgb.shape)

    dst_quad = np.float32([
        [0, 0],
        [bev_width - 1, 0],
        [bev_width - 1, bev_height - 1],
        [0, bev_height - 1],
    ])

    H = cv2.getPerspectiveTransform(src_quad, dst_quad)
    H_inv = np.linalg.inv(H)

    debug_rgb = img_rgb.copy()

    # Draw candidate lines.
    for l in candidates[:80]:
        x1, y1, x2, y2 = [int(v) for v in l]
        cv2.line(debug_rgb, (x1, y1), (x2, y2), (255, 255, 0), 2)

    # Draw road quad.
    q = src_quad.astype(int)
    cv2.polylines(debug_rgb, [q], isClosed=True, color=(255, 0, 0), thickness=4)
    for idx, p in enumerate(q):
        cv2.circle(debug_rgb, tuple(p), 7, (255, 0, 0), -1)
        cv2.putText(debug_rgb, str(idx), tuple(p + np.array([6, -6])), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    # Draw vanishing point if available and within drawable range.
    if vp is not None:
        vp_i = vp.astype(int)
        if -w <= vp_i[0] <= 2*w and -h <= vp_i[1] <= 2*h:
            cv2.circle(debug_rgb, tuple(vp_i), 9, (0, 255, 0), -1)
            cv2.putText(debug_rgb, 'VP', tuple(vp_i + np.array([8, 8])), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    result = {
        'H_image_to_bev': H,
        'H_bev_to_image': H_inv,
        'vanishing_point': vp,
        'road_quad_image': src_quad,
        'bev_size': (bev_width, bev_height),
        'num_candidate_lines': len(candidates),
        'num_intersections': int(len(intersections)),
        'debug_rgb': debug_rgb,
        'edges_roi': edges_roi,
    }
    return result


In [ ]:
#@title 5. Estimate homography and generate BEV preview

bev_width = 600 #@param {type:'integer'}
bev_height = 1000 #@param {type:'integer'}

result = estimate_auto_bev_homography(ref_rgb, bev_width=bev_width, bev_height=bev_height)
H = result['H_image_to_bev']

bev_rgb = cv2.warpPerspective(ref_rgb, H, (bev_width, bev_height))

print('Candidate lines:', result['num_candidate_lines'])
print('Line intersections:', result['num_intersections'])
print('Vanishing point:', result['vanishing_point'])
print('Road quad image points:')
print(result['road_quad_image'])
print('Homography image-to-BEV:')
print(H)

show_image(result['debug_rgb'], title='Debug: detected lines, vanishing point, road quad')
show_image(bev_rgb, title='Automatic BEV preview', figsize=(6, 10))


In [ ]:
#@title 6. Save homography, debug image, and BEV preview to Drive

ref_stem = ref_path.stem

homography_json_path = OUTPUT_DIR / f'{ref_stem}_homography.json'
debug_image_path = OUTPUT_DIR / f'{ref_stem}_debug_homography.jpg'
bev_image_path = OUTPUT_DIR / f'{ref_stem}_bev_auto.jpg'
edges_image_path = OUTPUT_DIR / f'{ref_stem}_edges_roi.png'

payload = {
    'reference_image': str(ref_path),
    'reference_image_name': ref_path.name,
    'image_shape': list(ref_rgb.shape),
    'bev_width': int(bev_width),
    'bev_height': int(bev_height),
    'H_image_to_bev': result['H_image_to_bev'].tolist(),
    'H_bev_to_image': result['H_bev_to_image'].tolist(),
    'vanishing_point': None if result['vanishing_point'] is None else result['vanishing_point'].tolist(),
    'road_quad_image': result['road_quad_image'].tolist(),
    'num_candidate_lines': int(result['num_candidate_lines']),
    'num_intersections': int(result['num_intersections']),
}

with open(homography_json_path, 'w') as f:
    json.dump(payload, f, indent=2)

save_rgb(debug_image_path, result['debug_rgb'])
save_rgb(bev_image_path, bev_rgb)
cv2.imwrite(str(edges_image_path), result['edges_roi'])

print('Saved homography JSON:', homography_json_path)
print('Saved debug image:', debug_image_path)
print('Saved BEV image:', bev_image_path)
print('Saved edges image:', edges_image_path)


## Optional tuning

The automatic quadrilateral is heuristic. If the BEV preview is too stretched, too narrow, or includes too much non-road area, tune the quadrilateral parameters below. This still remains unmanned once the parameters are fixed for a dataset/camera style.


In [ ]:
#@title 7. Optional: tune automatic road quadrilateral parameters

# Keep this cell optional. Run it only if the default BEV preview is poor.

top_y_ratio = 0.20 #@param {type:'number'}
bottom_y_ratio = 0.98 #@param {type:'number'}
bottom_left_ratio = 0.20 #@param {type:'number'}
bottom_right_ratio = 0.88 #@param {type:'number'}
top_width_ratio = 0.18 #@param {type:'number'}

vp = result['vanishing_point']
src_quad_tuned = build_road_quad_from_vp(
    vp,
    ref_rgb.shape,
    top_y_ratio=top_y_ratio,
    bottom_y_ratio=bottom_y_ratio,
    bottom_left_ratio=bottom_left_ratio,
    bottom_right_ratio=bottom_right_ratio,
    top_width_ratio=top_width_ratio,
)

dst_quad = np.float32([
    [0, 0],
    [bev_width - 1, 0],
    [bev_width - 1, bev_height - 1],
    [0, bev_height - 1],
])

H_tuned = cv2.getPerspectiveTransform(src_quad_tuned, dst_quad)
bev_tuned_rgb = cv2.warpPerspective(ref_rgb, H_tuned, (bev_width, bev_height))

debug_tuned = ref_rgb.copy()
q = src_quad_tuned.astype(int)
cv2.polylines(debug_tuned, [q], isClosed=True, color=(255, 0, 0), thickness=4)
for idx, p in enumerate(q):
    cv2.circle(debug_tuned, tuple(p), 7, (255, 0, 0), -1)
    cv2.putText(debug_tuned, str(idx), tuple(p + np.array([6, -6])), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

print('Tuned road quad image points:')
print(src_quad_tuned)
show_image(debug_tuned, title='Tuned road quad')
show_image(bev_tuned_rgb, title='Tuned BEV preview', figsize=(6, 10))

# Set this to True if you want to replace the saved default homography with the tuned one.
save_tuned_result = False #@param {type:'boolean'}

if save_tuned_result:
    tuned_payload = dict(payload)
    tuned_payload['H_image_to_bev'] = H_tuned.tolist()
    tuned_payload['H_bev_to_image'] = np.linalg.inv(H_tuned).tolist()
    tuned_payload['road_quad_image'] = src_quad_tuned.tolist()
    tuned_payload['tuned_parameters'] = {
        'top_y_ratio': float(top_y_ratio),
        'bottom_y_ratio': float(bottom_y_ratio),
        'bottom_left_ratio': float(bottom_left_ratio),
        'bottom_right_ratio': float(bottom_right_ratio),
        'top_width_ratio': float(top_width_ratio),
    }
    with open(homography_json_path, 'w') as f:
        json.dump(tuned_payload, f, indent=2)
    save_rgb(debug_image_path, debug_tuned)
    save_rgb(bev_image_path, bev_tuned_rgb)
    print('Overwrote saved homography with tuned version.')
else:
    print('Tuned result was not saved. Set save_tuned_result=True to save it.')


In [ ]:
#@title 8. Apply saved homography to several sample frames

with open(homography_json_path, 'r') as f:
    saved = json.load(f)

H_saved = np.array(saved['H_image_to_bev'], dtype=np.float32)
bev_size_saved = (int(saved['bev_width']), int(saved['bev_height']))

num_frames_to_warp = 6 #@param {type:'integer'}
num_frames_to_warp = max(1, min(num_frames_to_warp, len(images)))

preview_dir = OUTPUT_DIR / 'warped_sequence_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

for i, img_path in enumerate(images[:num_frames_to_warp]):
    img_rgb = imread_rgb(img_path)
    bev_rgb_i = cv2.warpPerspective(img_rgb, H_saved, bev_size_saved)
    out_path = preview_dir / f'{i:03d}_{img_path.stem}_bev.jpg'
    save_rgb(out_path, bev_rgb_i)
    print('Saved:', out_path)

# Show first and last warped previews.
first_preview = imread_rgb(sorted(preview_dir.glob('*.jpg'))[0])
last_preview = imread_rgb(sorted(preview_dir.glob('*.jpg'))[-1])
show_image(first_preview, title='First warped frame', figsize=(6, 10))
show_image(last_preview, title='Last warped frame', figsize=(6, 10))


## Outputs for Notebook 03

Notebook 03 can use these files:

```text
/Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_02_bev_homography/<frame>_homography.json
/Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_02_bev_homography/<frame>_bev_auto.jpg
/Users/vash/Dev/ResearchLab/Work/Drone3D/local_uav_bev_project/work/notebook_02_bev_homography/warped_sequence_preview/
```

Next notebook idea:

1. Load detection labels or run a vehicle detector.
2. Project detection bottom-centers into BEV using the saved homography.
3. Visualize vehicle positions in BEV.
